In [1]:
import cv2
from ultralytics import YOLO, solutions

In [9]:
model = YOLO('models/best.pt')

In [ ]:
import numpy as np
import cv2
from ultralytics import YOLO

cap = cv2.VideoCapture("data/sample_video.mp4")
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

video_writer = cv2.VideoWriter(
    "traffic_density_analysis.avi",
    cv2.VideoWriter_fourcc(*"mp4v"),  
    fps,
    (w, h),
)

region_points = {
    "region-left": [(465, 350), (609, 350), (510, 630), (2, 630)],
    "region-right": [(678, 350), (815, 350), (1203, 630), (743, 630)],
}

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect objects
    results = model(frame)[0]

    # Vẽ vùng quan sát
    for pts in region_points.values():
        cv2.polylines(frame, [np.array(pts)], isClosed=True, color=(0,255,0), thickness=2)

    # Vẽ box & label
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls = int(box.cls[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 2)
        cv2.putText(frame, f"{model.names[cls]} {conf:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

    # Hiển thị và ghi video
    cv2.imshow("Traffic Analysis", frame)
    video_writer.write(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()



0: 384x640 3 Vehicles, 76.7ms
Speed: 2.6ms preprocess, 76.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehicles, 88.6ms
Speed: 2.1ms preprocess, 88.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehicles, 69.9ms
Speed: 2.3ms preprocess, 69.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehicles, 66.1ms
Speed: 1.6ms preprocess, 66.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehicles, 64.6ms
Speed: 1.6ms preprocess, 64.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Vehicles, 71.9ms
Speed: 2.5ms preprocess, 71.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Vehicles, 74.4ms
Speed: 2.0ms preprocess, 74.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Vehicles, 74.5ms
Speed: 2.1ms preprocess, 74.5ms inference, 1.5ms postprocess per image at

In [13]:
from datetime import datetime
import pandas as pd

video_path = 'data/sample_video.mp4'
output_video_path = 'traffic_density_analysis.avi'
csv_log_path = 'traffic_log.csv'

heavy_traffic_threshold = 10 

vertices1 = np.array([(465, 350), (609, 350), (510, 630), (2, 630)], dtype=np.int32)
vertices2 = np.array([(678, 350), (815, 350), (1203, 630), (743, 630)], dtype=np.int32)

# Giới hạn lane (tách trái/phải)
lane_threshold = 609  

# Vị trí hiển thị thông tin trên frame
text_position_left_lane = (10, 50)
text_position_right_lane = (820, 50)
intensity_position_left_lane = (10, 100)
intensity_position_right_lane = (820, 100)

font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1
font_color = (255, 255, 255)
background_color = (0, 0, 255)

# ----------------- Khởi tạo video -----------------
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise Exception("Không thể mở video!")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# ----------------- Pipeline lưu trữ dữ liệu -----------------
traffic_log = []

# ----------------- Xử lý video -----------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    detection_frame = frame.copy()
    x1, x2 = 325, 635  # cắt vùng nếu cần
    detection_frame[:x1, :] = 0  
    detection_frame[x2:, :] = 0  

    # --- Detection ---
    results = model.predict(detection_frame, imgsz=640, conf=0.4)
    processed_frame = results[0].plot(line_width=1)

    # Khôi phục vùng ngoài vùng cắt
    processed_frame[:x1, :] = frame[:x1, :].copy()
    processed_frame[x2:, :] = frame[x2:, :].copy()

    # --- Vẽ polygon cho lanes ---
    cv2.polylines(processed_frame, [vertices1], isClosed=True, color=(0, 255, 0), thickness=2)
    cv2.polylines(processed_frame, [vertices2], isClosed=True, color=(255, 0, 0), thickness=2)

    # --- Đếm xe ---
    bounding_boxes = results[0].boxes
    vehicles_in_left_lane = 0
    vehicles_in_right_lane = 0

    for box in bounding_boxes.xyxy:
        if box[0] < lane_threshold:
            vehicles_in_left_lane += 1
        else:
            vehicles_in_right_lane += 1

    traffic_intensity_left = "Heavy" if vehicles_in_left_lane > heavy_traffic_threshold else "Smooth"
    traffic_intensity_right = "Heavy" if vehicles_in_right_lane > heavy_traffic_threshold else "Smooth"

    # --- Vẽ thông tin lên frame ---
    # Left lane
    cv2.rectangle(processed_frame, (text_position_left_lane[0]-10, text_position_left_lane[1]-25), 
                  (text_position_left_lane[0]+460, text_position_left_lane[1]+10), background_color, -1)
    cv2.putText(processed_frame, f'Vehicles in Left Lane: {vehicles_in_left_lane}', text_position_left_lane, 
                font, font_scale, font_color, 2, cv2.LINE_AA)

    cv2.rectangle(processed_frame, (intensity_position_left_lane[0]-10, intensity_position_left_lane[1]-25), 
                  (intensity_position_left_lane[0]+460, intensity_position_left_lane[1]+10), background_color, -1)
    cv2.putText(processed_frame, f'Traffic Intensity: {traffic_intensity_left}', intensity_position_left_lane, 
                font, font_scale, font_color, 2, cv2.LINE_AA)

    # Right lane
    cv2.rectangle(processed_frame, (text_position_right_lane[0]-10, text_position_right_lane[1]-25), 
                  (text_position_right_lane[0]+460, text_position_right_lane[1]+10), background_color, -1)
    cv2.putText(processed_frame, f'Vehicles in Right Lane: {vehicles_in_right_lane}', text_position_right_lane, 
                font, font_scale, font_color, 2, cv2.LINE_AA)

    cv2.rectangle(processed_frame, (intensity_position_right_lane[0]-10, intensity_position_right_lane[1]-25), 
                  (intensity_position_right_lane[0]+460, intensity_position_right_lane[1]+10), background_color, -1)
    cv2.putText(processed_frame, f'Traffic Intensity: {traffic_intensity_right}', intensity_position_right_lane, 
                font, font_scale, font_color, 2, cv2.LINE_AA)

    # --- Lưu frame ra video ---
    out.write(processed_frame)

    # --- Lưu dữ liệu thống kê ---
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)  # thời gian video tính bằng ms
    traffic_log.append({
        "timestamp_ms": timestamp,
        "vehicles_left": vehicles_in_left_lane,
        "intensity_left": traffic_intensity_left,
        "vehicles_right": vehicles_in_right_lane,
        "intensity_right": traffic_intensity_right
    })

    # --- Hiển thị video trực tiếp nếu muốn ---
    cv2.imshow("Traffic Analysis", processed_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ----------------- Giải phóng tài nguyên -----------------
cap.release()
out.release()
cv2.destroyAllWindows()

# ----------------- Lưu thống kê ra CSV -----------------
df = pd.DataFrame(traffic_log)
df.to_csv(csv_log_path, index=False)
print(f"Đã lưu dữ liệu thống kê giao thông vào {csv_log_path}")



0: 384x640 (no detections), 174.9ms
Speed: 7.6ms preprocess, 174.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 74.7ms
Speed: 2.5ms preprocess, 74.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 63.3ms
Speed: 2.6ms preprocess, 63.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 67.1ms
Speed: 1.6ms preprocess, 67.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 66.3ms
Speed: 1.8ms preprocess, 66.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 73.0ms
Speed: 2.6ms preprocess, 73.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 72.6ms
Speed: 1.5ms preprocess, 72.6ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 70.0ms
Speed: 2.5ms preprocess, 70.0ms